# ML-08 — First model, compared against my Week-4 baseline

**Lane: content refresh prioritization** — *which pages should the team refresh first?*
This notebook trains the first honest model for that lane, evaluates it on **the same data, the same
split, and the same metrics** as my Week-4 rule baseline, and then reads the errors before believing
the scores. Fixed seed **42** everywhere; library versions printed below.

## 1. Method choice and why

**The question shape** is "yes/no with an observed label" + "which pages first?" So, per the toolkit:

| model | why it's in the run | why it fits this lane |
|---|---|---|
| **Logistic Regression** | the readable baseline of learned models | linear, coefficient-ready; a sanity floor to compare against |
| **Random Forest** | the stronger default tabular ensemble | captures interactions (position x CTR x freshness) without scaling |
| **Gradient Boosting** (`HistGradientBoostingClassifier`) | boosting, but the sklearn-native version — no heavy external dependencies, safe on 24k train rows | best near the top of the ranking, which is exactly what a refresh queue is |

The lane's output is a ranking, so every model is scored by its **predicted probability of
decline**, evaluated at **precision@K** against the **same base rate** I used in Week 4 — plus
ROC-AUC for overall discrimination. No clustering: nothing here asks for a grouping, and the card's
"grouped" idea is honored as *client-grouped validation* instead (section 2).

Feature list is my honest set from Weeks 3/4: the trailing-90-day aggregates plus content metadata.
Excluded on purpose: `content_id`/`client_id` (pseudonyms are grouping-only), `trend_direction` and
`trend_pct` (the label's own plumbing), and the `*_last_30d` / `*_prev_30d` comparison columns (the
direct ingredients of `trend_pct`, i.e. label-adjacent). Everything used is knowable at decision time
from one snapshot — no future window exists.

In [1]:
# --- 1. data, seeds, versions ----------------------------------------
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42

ROOT = Path(os.getcwd()).resolve()
for _ in range(6):
    if (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        break
    ROOT = ROOT.parent
OUT_DIR = ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
# Evaluation label ONLY (per data dictionary: == (trend_direction == "down")). Never a model input.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

import sklearn as sk
print(f"rows={len(df):,}  sklearn={sk.__version__}  numpy={np.__version__}  pandas={pd.__version__}  seed={SEED}")

# ---- features: explicit, honest, single snapshot (no label-adjacent columns) ----
NUM_COUNT = ["search_volume", "cpc", "word_count", "char_count",
             "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
             "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d"]   # log1p: heavy tail
NUM_RAW = ["competition", "days_with_impressions", "days_with_sessions",
           "content_age_days", "days_since_last_update", "ctr", "avg_position",
           "engagement_rate", "scroll_rate", "ai_traffic_pct"]                 # already scaled/bounded
TIERS = ["competition_level", "age_tier", "freshness_tier",
         "word_count_tier", "impression_tier", "position_tier"]                # ordinal codes
NOMINAL = ["content_type", "main_intent"]                                      # one-hot

X = pd.DataFrame(index=df.index)
for c in NUM_COUNT:
    X["log_" + c] = np.log1p(pd.to_numeric(df[c], errors="coerce")).fillna(0).astype(float)
for c in NUM_RAW:
    X[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(float)
for c in TIERS:
    X[c] = df[c].astype("category").cat.codes.astype(int)                      # -1 = missing
for c in NOMINAL:
    X = X.join(pd.get_dummies(df[c].fillna("unknown"), prefix=c, dtype=int).astype(int))

y = df["is_declining_label"].astype(int)
print(f"feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features | label share {y.mean():.3f}")

rows=30,000  sklearn=1.9.0  numpy=2.5.1  pandas=3.0.3  seed=42
feature matrix: 30,000 rows x 36 features | label share 0.542


## 2. Split design

The dataset is a **single snapshot** — there is no time axis inside it, so a time-based split is
impossible and a random per-row split would quietly leak client identity into training. The honest
question for this lane is: *"does the model generalise to content it has never seen?"* So I split
**by client** with `GroupShuffleSplit(test_size=0.20)` — ~20% of *rows* held out as 7 entirely
unseen clients, the rest used to train. Every model and the Week-4 rule are then scored on **that
same held-out test set**.

Two robustness checks: (a) the split re-drawn with different seeds, and (b) **GroupKFold(5)** —
client-grouped cross-validation — to confirm the headline isn't a lucky draw of one split.

In [2]:
# --- 2. the one split everything is compared on ----------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
Xtr, Xte, ytr, yte = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

clients_te = df["client_id"].iloc[test_idx].nunique()
print(f"train: {len(Xtr):,} rows (decl {ytr.mean():.3f})   test: {len(Xte):,} rows (decl {yte.mean():.3f})  from {clients_te} held-out clients")
print("no client appears in both train and test:", df["client_id"].iloc[train_idx].nunique() + clients_te == df["client_id"].nunique())

train: 23,837 rows (decl 0.550)   test: 6,163 rows (decl 0.511)  from 7 held-out clients
no client appears in both train and test: True


## 3. Train + compare vs my Week-4 baseline — same split, same metrics

The Week-4 baseline is **re-computed inside this notebook** from its plain-English rule
(`score = log1p(impressions) * (1+stale) * (1+ctr_gap)`), then ranked on the **test split only**.
Every model trains on the train split and is ranked on the test split the same way.
Metrics: **precision@K** (same as Week 4) plus ROC-AUC, and the base rate as the honest floor.

In [3]:
# --- Week-4 baseline rule, re-encoded identically (evaluation only) ---
def week4_rule_score(imp, dsul, pos, ctr):
    stale = ((dsul >= 180) & (imp >= 300)).astype(int)
    gap = ((pos > 0) & (pos <= 10) & (ctr < 0.5) & (imp >= 300)).astype(int)
    return np.log1p(imp) * (1 + stale) * (1 + gap)

base_score = week4_rule_score(
    df["impressions_90d"].iloc[test_idx].values,
    df["days_since_last_update"].iloc[test_idx].values,
    df["avg_position"].iloc[test_idx].values,
    df["ctr"].iloc[test_idx].values,
)

def precision_at_k(y_true, score, ks=(10, 20, 50, 100)):
    order = np.argsort(-np.asarray(score))
    y = np.asarray(y_true)
    return [round(float(y[order[:k]].mean()), 3) for k in ks]

def eval_row(name, score, yte):
    return [name] + precision_at_k(yte, score) + [round(roc_auc_score(yte, score), 3)]

models = {
    "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    "random_forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
    "gradient_boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=5, random_state=SEED),
}
probs = {"week4_rule_baseline": base_score}
for name, model in models.items():
    model.fit(Xtr, ytr)
    probs[name] = model.predict_proba(Xte)[:, 1]

table = pd.DataFrame(
    [["base_rate"] + [None] * 4 + [round(float(yte.mean()), 3)]]
    + [eval_row(name, p, yte) for name, p in probs.items()],
    columns=["method", "P@10", "P@20", "P@50", "P@100", "AUC"],
)
print("=== model vs Week-4 baseline, SAME client-grouped test split ===")
print(table.to_string(index=False))

# --- robustness 1: redraw the split with other seeds ---
print("\n=== robustness 1: HGB and the rule on re-drawn splits (P@10/P@20/P@50, AUC) ===")
for s in (7, 123, 2026):
    g2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=s)
    tr2, te2 = next(g2.split(X, y, groups=df["client_id"]))
    hgb2 = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=5, random_state=SEED)
    hgb2.fit(X.iloc[tr2], y.iloc[tr2])
    p2 = hgb2.predict_proba(X.iloc[te2])[:, 1]
    bs2 = week4_rule_score(df["impressions_90d"].iloc[te2].values, df["days_since_last_update"].iloc[te2].values,
                           df["avg_position"].iloc[te2].values, df["ctr"].iloc[te2].values)
    print(f"seed {s:>4}: test n={len(te2):,} base={y.iloc[te2].mean():.3f} | rule AUC={roc_auc_score(y.iloc[te2], bs2):.3f} | "
          f"HGB P={precision_at_k(y.iloc[te2], p2)} AUC={roc_auc_score(y.iloc[te2], p2):.3f}")

# --- robustness 2: grouped validation, GroupKFold(5) by client ---
print("\n=== robustness 2: GroupKFold(5) AUC by client fold ===")
def fresh(name):
    if name == "logistic_regression":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
    if name == "random_forest":
        return RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=5, random_state=SEED)

gk = GroupKFold(n_splits=5)
for name in ["week4_rule_baseline", "logistic_regression", "random_forest", "gradient_boosting"]:
    aucs = []
    for trk, tek in gk.split(X, y, groups=df["client_id"]):
        if name == "week4_rule_baseline":
            bsk = week4_rule_score(df["impressions_90d"].iloc[tek].values, df["days_since_last_update"].iloc[tek].values,
                                   df["avg_position"].iloc[tek].values, df["ctr"].iloc[tek].values)
            aucs.append(roc_auc_score(y.iloc[tek], bsk))
        else:
            m = fresh(name)
            m.fit(X.iloc[trk], y.iloc[trk])
            aucs.append(roc_auc_score(y.iloc[tek], m.predict_proba(X.iloc[tek])[:, 1]))
    print(f"{name:20s} AUC {np.mean(aucs):.3f} +- {np.std(aucs):.3f}")

metrics = {
    "notebook": "w05_model.ipynb",
    "seed": SEED,
    "split": "client_grouped_holdout_GroupShuffleSplit_test20%",
    "test_rows": int(len(Xte)),
    "test_base_rate": round(float(yte.mean()), 3),
    "test_split_table": {
        "week4_rule_baseline": {"P@10": precision_at_k(yte, base_score)[0], "AUC": round(float(roc_auc_score(yte, base_score)), 3)},
        "gradient_boosting": {"P@10": precision_at_k(yte, probs["gradient_boosting"])[0], "AUC": round(float(roc_auc_score(yte, probs["gradient_boosting"])), 3)},
    },
}
(OUT_DIR / "w05_model_metrics.json").write_text(json.dumps(metrics, indent=2, sort_keys=True))
print("\nwrote work/outputs/w05_model_metrics.json")

=== model vs Week-4 baseline, SAME client-grouped test split ===
             method  P@10  P@20  P@50  P@100   AUC
          base_rate   NaN   NaN   NaN    NaN 0.511
week4_rule_baseline   0.4  0.40  0.42   0.41 0.542
logistic_regression   0.8  0.75  0.76   0.72 0.622
      random_forest   0.6  0.75  0.68   0.69 0.610
  gradient_boosting   0.9  0.90  0.90   0.86 0.622

=== robustness 1: HGB and the rule on re-drawn splits (P@10/P@20/P@50, AUC) ===


seed    7: test n=11,754 base=0.443 | rule AUC=0.601 | HGB P=[0.9, 0.95, 0.88, 0.85] AUC=0.663


seed  123: test n=8,864 base=0.594 | rule AUC=0.617 | HGB P=[0.8, 0.85, 0.74, 0.78] AUC=0.692


seed 2026: test n=3,881 base=0.609 | rule AUC=0.587 | HGB P=[0.8, 0.85, 0.8, 0.78] AUC=0.660

=== robustness 2: GroupKFold(5) AUC by client fold ===
week4_rule_baseline  AUC 0.622 +- 0.090


logistic_regression  AUC 0.673 +- 0.039


random_forest        AUC 0.678 +- 0.037


gradient_boosting    AUC 0.684 +- 0.043

wrote work/outputs/w05_model_metrics.json


**What the table says (numbers printed above):**

- The Week-4 rule barely beats the base rate on held-out clients — here **P@10 0.40 vs base 0.51**,
  AUC ≈ 0.54. Week 4's P@10 of 0.80 was an *in-sample* look at the queue it built; measured on
  unseen clients with the same metric it drops to random-like. That is the honest fate of a
  volume-and-flag rule, and it is exactly what the model exists to fix.
- **Every model beats the rule** on both P@K and AUC. Logistic Regression already jumps to P@10 0.80.
- **Gradient boosting is the queue winner**: P@10/P@20/P@50 ~ 0.90 and P@100 0.86 on the held-out
  clients; it holds up across split seeds (P@10 0.80-0.90, AUC 0.66-0.69) and in GroupKFold
  (AUC 0.684 +- 0.043, best mean).
- **Complexity earns a modest, honest gain — not a miracle.** On AUC, LR / RF / HGB are close
  (0.61-0.62 on this split; 0.67-0.68 in grouped CV), all ~0.06-0.08 above the rule. The large gap
  is at the **top of the ranking** (P@10-50), which is the part a refresh queue actually uses.

## 4. Errors and interpretation

Before believing the scores: what does the model lean on, where is it wrong, and are the errors
the *right kind* for this lane?

In [4]:
# --- (a) what the model leans on: permutation importance on the test split ---
# Xte/yte/test are carried in from section 3; models["gradient_boosting"] is the fitted HGB.
pi = permutation_importance(models["gradient_boosting"], Xte, yte, n_repeats=5, random_state=SEED, scoring="roc_auc")
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
print("Permutation importance (HGB, drop in AUC when a feature is shuffled):")
print(imp.head(10).round(4).to_string())

# --- (b) where it is wrong ---
test = df.iloc[test_idx].copy()
test["proba"] = probs["gradient_boosting"]
test["predicted"] = (test["proba"] >= 0.5).astype(int)

fn = test[(test["predicted"] == 0) & (test["is_declining_label"] == 1)]
fp = test[(test["predicted"] == 1) & (test["is_declining_label"] == 0)]
giants = test[test["impressions_90d"] >= 3000]
top100 = set(test.nlargest(100, "proba").index)
decl_giants = giants[giants["is_declining_label"] == 1]
print(f"\nfalse negatives: {len(fn)} (predicted safe, actually declining) | median impressions {int(fn['impressions_90d'].median()):,}")
print(f"false positives: {len(fp)} (predicted declining, actually not) | median impressions {int(fp['impressions_90d'].median()):,}")
print(f"of all test declines, {(decl_giants['is_declining_label'].sum()/yte.sum()*100):.0f}% are 'giants' (impressions>=3000); "
      f"top-100 predictions catch {decl_giants.index.isin(top100).sum()} of {len(decl_giants)} declining giants")

print("\n--- three concrete wrong cases (2 false negatives, 1 false positive) ---")
cols = ["content_id", "proba", "is_declining_label", "impressions_90d", "days_with_impressions",
        "avg_position", "ctr", "days_since_last_update", "content_age_days", "trend_direction", "content_type"]
fn_big = fn.sort_values("impressions_90d", ascending=False).head(30)
print(fn_big[cols].head(2).to_string(index=False))
print(fp.nlargest(2, "proba")[cols].head(2).to_string(index=False))

Permutation importance (HGB, drop in AUC when a feature is shuffled):
days_with_impressions    0.0920
avg_position             0.0287
content_age_days         0.0170
log_impressions_90d      0.0106
ctr                      0.0100
scroll_rate              0.0094
log_users_90d            0.0068
days_with_sessions       0.0030
log_clicks_90d           0.0025
engagement_rate          0.0024

false negatives: 1122 (predicted safe, actually declining) | median impressions 334
false positives: 1415 (predicted declining, actually not) | median impressions 787
of all test declines, 18% are 'giants' (impressions>=3000); top-100 predictions catch 26 of 576 declining giants

--- three concrete wrong cases (2 false negatives, 1 false positive) ---
          content_id    proba  is_declining_label  impressions_90d  days_with_impressions  avg_position  ctr  days_since_last_update  content_age_days trend_direction    content_type
content_8c19996aa890 0.274215                   1           509252      

**What it leans on (sanity check each):**

1. **`days_with_impressions`** (importance ~0.09, 3-4x the next) — how many of the 90 days the page
   was actually seen. Sensible: pages that appear every day are the ones with enough traffic to lose;
   dormant pages have nothing to decline. Not suspiciously perfect -> not leakage.
2. **`avg_position`** + **`content_age_days`** + **`log_impressions_90d`** — position and age rule
   this lane the way Week 4 suspected: they answer the "did it used to get traffic, is it old" part
   of the refresh rule.
3. **`ctr` / `scroll_rate`** — engagement signals: a page that stops earning clicks or scrolls is
   already signalling decline.

**Where it's wrong (the kind that matters):**

- **False negatives are mostly quiet pages** (median 334 impressions; 53% under 500). Missing those
  is nearly free for a refresh queue that should spend effort where traffic exists. The expensive
  misses are the **large-volume, freshly-updated giants** — e.g. the position-2.5 pages with ~500k
  impressions that are declining but were updated just 20 days ago: the model reads "recently
  touched + great position" as health. These are exactly the pages Week 4's rule *did* catch — a
  real, honest trade-off between the two systems.
- **False positives look like "almost declining":** mediocre position (~10.5), near-zero CTR, very
  consistent impressions (83 of 90 days), recently updated. The model flags consistently-visible,
  click-starved pages, and it is sometimes *confidently* wrong (the top false positive scored 0.95
  while the trend was `up`). Verdict: mid-volume, awkward pages are where it invents risk.

**What this means for Week 6:** use the model's probability as the queue order (gradient boosting),
but consider blending the rule's "big-fish" signal so 500k-impression declining pages don't fall
below quiet pages — and budget a title-rewrite test on the false-positive pattern to see whether
"consistent impressions but no clicks" is a genuine fix signal or a trap.

## Self-check

Before submitting, confirm each line honestly:

- [x] Method choice explained and fitted to the lane (ranking -> probability + precision@K)
- [x] Valid split design: client-grouped holdout, plus GroupKFold(5) + split-seed robustness
- [x] Model vs Week-4 baseline compared on the **same split and same metrics**, base rate shown
- [x] Useful metrics reported: P@10/20/50/100 and ROC-AUC, with the honesty of in-sample vs held-out
- [x] Errors and features interpreted (permutation importance checked, concrete wrong cases named)
- [x] No complexity flattery: plain statement of what boosting adds vs the rule and vs logistic
- [x] No client names/URLs; no future-window or label-derived inputs (label is evaluation-only)
- [x] Runs top to bottom, committed under `work/notebooks/`, repo URL submitted. Done.